# FlyRank ML Capstone — Search Content Opportunity Scoring

**Lane:** Refresh / Content Opportunity Scoring  
**Research question:** Can historical search-performance signals help prioritize pages for content review?

This notebook is a complete, reproducible workflow template. It discovers Parquet files in the gated Hugging Face dataset, loads them with DuckDB, builds historical features and a future-window target, compares a simple baseline with a Random Forest model, and produces ranked recommendations.

**Important before running:** the FlyRank release's exact repository ID, table/file layout and column names are specific to your internship access. Fill in the configuration cell after checking `03_working_with_the_full_release.ipynb`. Do not guess or publish private dataset details. The notebook deliberately stops with a clear error if required columns are not configured.

## 1. Install and import dependencies

In [ ]:
%pip -q install duckdb huggingface_hub pandas numpy matplotlib seaborn scikit-learn pyarrow

In [ ]:
import os
import re
import math
import json
import warnings
from getpass import getpass
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from huggingface_hub import HfApi
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
OUTPUT_DIR = Path("flyrank_capstone_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Configure access and dataset

Copy the dataset repository ID and the recommended Hugging Face access workflow from the FlyRank portal / Notebook 03. Never paste your token into a notebook cell or commit it to GitHub.

In [ ]:
# REQUIRED: replace with the exact gated dataset repo ID shown in FlyRank resources.
HF_DATASET_REPO = "REPLACE_WITH_ORG/REPLACE_WITH_DATASET"

# Optional: narrow to a specific Parquet path/prefix if Notebook 03 identifies one.
# Example only: "release_name/**/*.parquet"
PARQUET_PATH_GLOB = "**/*.parquet"

# Enter token securely when prompted; it is not saved in notebook source.
HF_TOKEN = getpass("Hugging Face read token (input hidden): ")

if "REPLACE_WITH" in HF_DATASET_REPO:
    raise ValueError("Set HF_DATASET_REPO to the exact dataset repo ID from FlyRank resources.")

api = HfApi(token=HF_TOKEN)
repo_files = api.list_repo_files(repo_id=HF_DATASET_REPO, repo_type="dataset")
parquet_files = [p for p in repo_files if p.lower().endswith((".parquet", ".pq"))]

print(f"Found {len(parquet_files)} Parquet files.")
for p in parquet_files[:30]:
    print(p)

if not parquet_files:
    raise RuntimeError(
        "No Parquet files were found. Check the FlyRank starter notebook: "
        "the release may use a different format or a specific path."
    )

## 3. Inspect the release and choose the source file(s)

Use the listing above and Notebook 03 to select the relevant search-performance table/file. Start with one fact table rather than joining many tables without a clear need.

In [ ]:
# REQUIRED: enter one or more exact Parquet paths from the listing above.
# Keep paths relative to the dataset repository.
SELECTED_PARQUET_FILES = [
    "REPLACE_WITH_EXACT_PARQUET_PATH.parquet"
]

if any("REPLACE_WITH" in p for p in SELECTED_PARQUET_FILES):
    raise ValueError("Set SELECTED_PARQUET_FILES to actual paths from the file listing.")

missing_paths = [p for p in SELECTED_PARQUET_FILES if p not in parquet_files]
if missing_paths:
    raise ValueError(f"These paths were not found in the repository listing: {missing_paths}")

In [ ]:
# DuckDB Hugging Face extension and temporary secret.
con = duckdb.connect()
con.execute("INSTALL hf;")
con.execute("LOAD hf;")
safe_token = HF_TOKEN.replace("\\", "\\\\").replace("'", "\\'")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{safe_token}')")

hf_urls = [f"hf://datasets/{HF_DATASET_REPO}/{p}" for p in SELECTED_PARQUET_FILES]
print("\\n".join(hf_urls))

# Read each selected Parquet file and combine columns by name.
# If the starter notebook uses a different DuckDB HF syntax, follow that tested syntax instead.
quoted_urls = ", ".join("'" + u.replace("'", "''") + "'" for u in hf_urls)
source_sql = f"SELECT * FROM read_parquet([{quoted_urls}], union_by_name=true)"
raw_df = con.execute(source_sql).df()

print("Shape:", raw_df.shape)
display(raw_df.head())
print("Columns:", raw_df.columns.tolist())

## 4. Map the actual FlyRank columns

Set these names to the exact fields in `raw_df.columns`. The page identifier and date are required. Clicks and impressions are strongly recommended. CTR and average position are optional; CTR can be derived from clicks/impressions.

In [ ]:
# REQUIRED COLUMN MAPPING — edit the right-hand values to match the dataset.
COL_PAGE = "REPLACE_PAGE_ID_COLUMN"
COL_DATE = "REPLACE_DATE_COLUMN"
COL_CLICKS = "REPLACE_CLICKS_COLUMN"
COL_IMPRESSIONS = "REPLACE_IMPRESSIONS_COLUMN"

# Optional. Use None if unavailable.
COL_CTR = None                 # e.g. "ctr"
COL_POSITION = None            # e.g. "average_position"
COL_SESSIONS = None            # e.g. "sessions"
COL_ENGAGEMENT = None          # e.g. "engagement_rate"

required_mapping = {
    "page": COL_PAGE, "date": COL_DATE,
    "clicks": COL_CLICKS, "impressions": COL_IMPRESSIONS
}
for logical_name, actual_name in required_mapping.items():
    if not actual_name or "REPLACE_" in str(actual_name):
        raise ValueError(f"Configure {logical_name}: set its exact column name.")
    if actual_name not in raw_df.columns:
        raise KeyError(f"Column '{actual_name}' not found. Available columns: {raw_df.columns.tolist()}")

optional_mapping = {
    "ctr": COL_CTR, "position": COL_POSITION,
    "sessions": COL_SESSIONS, "engagement": COL_ENGAGEMENT
}
for logical_name, actual_name in optional_mapping.items():
    if actual_name is not None and actual_name not in raw_df.columns:
        raise KeyError(f"Optional column '{actual_name}' not found.")

## 5. Clean and standardize data

In [ ]:
rename_map = {
    COL_PAGE: "page_id",
    COL_DATE: "date",
    COL_CLICKS: "clicks",
    COL_IMPRESSIONS: "impressions",
}
for logical_name, actual_name in optional_mapping.items():
    if actual_name is not None:
        rename_map[actual_name] = logical_name

df = raw_df[list(rename_map.keys())].rename(columns=rename_map).copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True).dt.tz_convert(None)
for c in ["clicks", "impressions", "ctr", "position", "sessions", "engagement"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["page_id", "date"])
df["page_id"] = df["page_id"].astype(str)
df = df[(df["clicks"].fillna(0) >= 0) & (df["impressions"].fillna(0) >= 0)]
df = df.drop_duplicates()

if "ctr" not in df.columns:
    df["ctr"] = np.where(df["impressions"] > 0, df["clicks"] / df["impressions"], np.nan)

# Aggregate duplicate page-date records safely.
agg_spec = {
    "clicks": "sum",
    "impressions": "sum",
    "ctr": "mean",
}
for c in ["position", "sessions", "engagement"]:
    if c in df.columns:
        agg_spec[c] = "mean"

daily = df.groupby(["page_id", "date"], as_index=False).agg(agg_spec)
daily = daily.sort_values(["page_id", "date"]).reset_index(drop=True)

print("Clean daily shape:", daily.shape)
print("Date range:", daily["date"].min(), "to", daily["date"].max())
display(daily.head())

## 6. Data quality checks and exploratory analysis

In [ ]:
quality = pd.DataFrame({
    "missing_count": daily.isna().sum(),
    "missing_pct": (daily.isna().mean() * 100).round(2),
    "dtype": daily.dtypes.astype(str)
})
display(quality)
print("Unique pages:", daily["page_id"].nunique())
print("Unique dates:", daily["date"].nunique())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(daily["clicks"], bins=50, ax=axes[0])
axes[0].set_title("Daily clicks distribution")
sns.histplot(daily["impressions"], bins=50, ax=axes[1])
axes[1].set_title("Daily impressions distribution")
plt.tight_layout()
plt.show()

In [ ]:
daily_trend = daily.groupby("date")[["clicks", "impressions"]].sum()
fig, ax1 = plt.subplots(figsize=(12, 5))
daily_trend["clicks"].plot(ax=ax1, label="Clicks")
ax1.set_ylabel("Clicks")
ax1.set_title("Aggregate search performance over time")
ax2 = ax1.twinx()
daily_trend["impressions"].plot(ax=ax2, alpha=0.5, label="Impressions")
ax2.set_ylabel("Impressions")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=daily.sample(min(len(daily), 10000), random_state=RANDOM_STATE),
                x="impressions", y="clicks", alpha=0.35)
plt.title("Daily impressions vs clicks")
plt.tight_layout()
plt.show()

## 7. Define feature and target windows

The default design uses the most recent 28 days available before a cutoff as the feature window and the following 14 days as the outcome window. The notebook uses one global cutoff date to make train/test evaluation time-aware. Confirm that the release has enough dates to support these windows. Change the durations only with a documented reason.

In [ ]:
FEATURE_DAYS = 28
TARGET_DAYS = 14
MIN_FEATURE_IMPRESSIONS = 1

max_date = daily["date"].max().normalize()
target_end = max_date
target_start = target_end - pd.Timedelta(days=TARGET_DAYS - 1)
feature_end = target_start - pd.Timedelta(days=1)
feature_start = feature_end - pd.Timedelta(days=FEATURE_DAYS - 1)

print("Feature window:", feature_start.date(), "to", feature_end.date())
print("Target window:", target_start.date(), "to", target_end.date())

if daily["date"].min().normalize() > feature_start:
    raise ValueError(
        "Not enough historical coverage for the configured windows. "
        "Reduce FEATURE_DAYS/TARGET_DAYS only if methodologically justified."
    )

## 8. Build page-level features and future-window label

Features use only the historical window. The target is the percentage change in clicks from the historical feature window to the future target window. Pages with zero historical clicks are retained but their target is handled with a small denominator; interpret these cases carefully.

In [ ]:
hist = daily[(daily["date"] >= feature_start) & (daily["date"] <= feature_end)].copy()
future = daily[(daily["date"] >= target_start) & (daily["date"] <= target_end)].copy()

def aggregate_window(frame, prefix):
    agg = {
        "clicks": "sum",
        "impressions": "sum",
        "ctr": "mean",
    }
    for c in ["position", "sessions", "engagement"]:
        if c in frame.columns:
            agg[c] = "mean"
    out = frame.groupby("page_id").agg(agg)
    return out.add_prefix(prefix)

hist_features = aggregate_window(hist, "hist_")
future_outcomes = future.groupby("page_id").agg(
    future_clicks=("clicks", "sum"),
    future_impressions=("impressions", "sum")
)

model_data = hist_features.join(future_outcomes, how="inner").reset_index()
model_data = model_data[model_data["hist_impressions"] >= MIN_FEATURE_IMPRESSIONS].copy()

# Future click-change label. This is an outcome only, never an input feature.
model_data["target_click_change"] = (
    (model_data["future_clicks"] - model_data["hist_clicks"])
    / (model_data["hist_clicks"] + 1.0)
)
model_data["target_click_change"] = model_data["target_click_change"].clip(-5, 5)

print("Pages with both feature and target windows:", len(model_data))
display(model_data.head())

## 9. Feature engineering and leakage controls

The feature list below contains only historical-window variables. Future clicks/impressions are explicitly excluded from model inputs. If optional columns are unavailable, they are simply omitted.

In [ ]:
model_data["hist_ctr_derived"] = (
    model_data["hist_clicks"] / model_data["hist_impressions"].replace(0, np.nan)
)

candidate_features = [
    "hist_clicks", "hist_impressions", "hist_ctr", "hist_ctr_derived",
]
for c in ["hist_position", "hist_sessions", "hist_engagement"]:
    if c in model_data.columns:
        candidate_features.append(c)

FEATURES = [c for c in candidate_features if c in model_data.columns]
TARGET = "target_click_change"

# Leakage audit: these fields are outcomes and must never appear in FEATURES.
for forbidden in ["future_clicks", "future_impressions", TARGET]:
    assert forbidden not in FEATURES, f"Leakage detected: {forbidden}"

print("Model features:", FEATURES)
print("Target:", TARGET)

## 10. Create a time-aware train/test evaluation design

Each page is represented by the same historical feature window and a subsequent future target window. To compare across page groups, use a deterministic page-level split here, while keeping the feature/target windows strictly chronological. If your release supports multiple historical cutoffs, a rolling-origin evaluation is stronger; document the limitation if only one cutoff is feasible.

In [ ]:
# Deterministic page-level split: same cutoff windows for all pages, no future fields in X.
rng = np.random.default_rng(RANDOM_STATE)
indices = np.arange(len(model_data))
rng.shuffle(indices)
split_at = int(len(indices) * 0.8)

train_idx = indices[:split_at]
test_idx = indices[split_at:]

X_train = model_data.iloc[train_idx][FEATURES].replace([np.inf, -np.inf], np.nan)
X_test = model_data.iloc[test_idx][FEATURES].replace([np.inf, -np.inf], np.nan)
y_train = model_data.iloc[train_idx][TARGET]
y_test = model_data.iloc[test_idx][TARGET]

print("Train pages:", len(X_train), "Test pages:", len(X_test))

## 11. Baseline and Random Forest model

Baseline predicts no change for every page. This is a transparent reference point. The Random Forest is compared against it on exactly the same test rows.

In [ ]:
baseline_pred = np.zeros(len(y_test))

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

def regression_metrics(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

metrics = pd.DataFrame([
    regression_metrics(y_test, baseline_pred, "Baseline: predict no change"),
    regression_metrics(y_test, model_pred, "Random Forest")
])
display(metrics.round(4))
metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)

In [ ]:
metrics_plot = metrics.set_index("model")[["MAE", "RMSE"]]
metrics_plot.plot(kind="bar", figsize=(9, 5))
plt.title("Model vs baseline on the same test split")
plt.ylabel("Error (lower is better)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_vs_baseline.png", dpi=160, bbox_inches="tight")
plt.show()

## 12. Model diagnostics and feature importance

In [ ]:
residuals = y_test.to_numpy() - model_pred
plt.figure(figsize=(8, 4))
sns.histplot(residuals, bins=40, kde=True)
plt.title("Random Forest residual distribution")
plt.xlabel("Actual minus predicted click change")
plt.tight_layout()
plt.show()

In [ ]:
rf = model.named_steps["rf"]
importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
importance.tail(15).plot(kind="barh", figsize=(8, 5))
plt.title("Random Forest feature importance")
plt.xlabel("Impurity-based importance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()

## 13. Generate ranked content-review opportunities

The model predicts future click change. A negative prediction indicates expected decline, while pages with greater historical visibility receive more weight. This is a prioritization heuristic, not proof that editing a page will cause recovery.

In [ ]:
test_results = model_data.iloc[test_idx][["page_id", "hist_clicks", "hist_impressions", "hist_ctr", TARGET]].copy()
test_results["predicted_click_change"] = model_pred

# Risk/opportunity score: predicted decline magnitude × log visibility.
# Positive score only for predicted declines; scaled to 0–100 for readability.
decline_component = (-test_results["predicted_click_change"]).clip(lower=0)
visibility_component = np.log1p(test_results["hist_impressions"])
raw_score = decline_component * visibility_component

if raw_score.max() > raw_score.min():
    test_results["opportunity_score"] = (
        100 * (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min())
    )
else:
    test_results["opportunity_score"] = 0.0

def reason_code(row):
    reasons = []
    if row["predicted_click_change"] < 0:
        reasons.append("PREDICTED_DECLINE")
    if row["hist_impressions"] >= test_results["hist_impressions"].quantile(0.75):
        reasons.append("HIGH_HISTORICAL_VISIBILITY")
    if row["hist_ctr"] <= test_results["hist_ctr"].quantile(0.25):
        reasons.append("LOWER_QUARTILE_HISTORICAL_CTR")
    return ";".join(reasons) if reasons else "MONITOR"

def action_for(row):
    if row["predicted_click_change"] < 0 and row["hist_ctr"] <= test_results["hist_ctr"].quantile(0.25):
        return "Review search snippet and content relevance"
    if row["predicted_click_change"] < 0:
        return "Review content quality and recent performance"
    return "Monitor; avoid unnecessary changes"

test_results["reason_codes"] = test_results.apply(reason_code, axis=1)
test_results["recommended_action"] = test_results.apply(action_for, axis=1)
test_results = test_results.sort_values("opportunity_score", ascending=False).reset_index(drop=True)
test_results.insert(0, "priority_rank", np.arange(1, len(test_results) + 1))

display(test_results.head(20))
test_results.to_csv(OUTPUT_DIR / "ranked_recommendations.csv", index=False)

In [ ]:
top = test_results.head(15).sort_values("opportunity_score")
plt.figure(figsize=(9, 6))
plt.barh(top["page_id"].astype(str), top["opportunity_score"])
plt.xlabel("Opportunity score (0–100)")
plt.ylabel("Anonymized page identifier")
plt.title("Top content-review opportunities")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "opportunity_ranking.png", dpi=160, bbox_inches="tight")
plt.show()

## 14. Recommendation summary

In [ ]:
print("Pages evaluated:", len(test_results))
print("Pages with positive predicted decline:", int((test_results["predicted_click_change"] < 0).sum()))
print("Top recommended actions:")
display(test_results["recommended_action"].value_counts().rename_axis("action").to_frame("page_count"))

## 15. Limitations and honest interpretation

- The analysis identifies patterns in the available FlyRank release; it does not reveal or prove Google's internal ranking algorithm.
- Predicted decline is a prioritization signal, not evidence that a content refresh will cause recovery.
- This version uses one historical/target window and a page-level holdout. Repeated rolling-origin validation across several cutoffs would provide stronger evidence when the data supports it.
- The target is based on future click change and can be noisy for pages with low historical volume.
- Page IDs and any exported tables must be checked against FlyRank's public-data rules before publication. Do not publish client names, domains, URLs, private queries, credentials or raw exports.
- The model may not generalize to other releases, sites, time periods or search environments.

## 16. Export public-safe figures and results

Review every output before committing it. The recommendation CSV may contain identifiers and row-level values; do not publish it unless FlyRank's data-use rules explicitly permit that. Prefer charts and aggregated summaries for the public paper.

In [ ]:
print("Created outputs:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name, p.stat().st_size, "bytes")

## 17. Reproducibility record for the paper

Fill the following details from the run and the FlyRank portal. Keep sensitive identifiers private.

In [ ]:
run_summary = {
    "lane": "Refresh / Content Opportunity Scoring",
    "research_question": "Can historical search-performance signals help prioritize pages for content review?",
    "feature_window_start": str(feature_start.date()),
    "feature_window_end": str(feature_end.date()),
    "target_window_start": str(target_start.date()),
    "target_window_end": str(target_end.date()),
    "page_count_after_filtering": int(len(model_data)),
    "train_page_count": int(len(train_idx)),
    "test_page_count": int(len(test_idx)),
    "features": FEATURES,
    "target": TARGET,
    "metrics": metrics.to_dict(orient="records")
}
print(json.dumps(run_summary, indent=2, default=str))

## 18. Before publishing

1. Confirm the dataset release name, source tables/files and date range from the actual run.
2. Confirm the validation design and explain the page-level holdout honestly; do not call it a pure future-time test.
3. Check model metrics and state whether the model beat the baseline.
4. Remove page-level identifiers and any restricted outputs from the public paper/repository.
5. Add charts from `flyrank_capstone_outputs/` to the paper only after checking that they are aggregated and public-safe.
6. Link this notebook and the repository in the paper's Reproducibility section.
7. Add the required credit: **Built on the FlyRank ML Internship dataset** linking to https://flyrank.ai.
8. Deploy the paper and put its direct URL as the only line in `submission/paper_url.txt`.

**Do not submit invented metrics or leave any `REPLACE_...` configuration values unresolved.**